# Track C: Instructional Assistant — LoRA Fine-tuning + LangChain Tools

**Модель:** `Qwen/Qwen3-1.7B` (BASE)

**Подход:** двухэтапное обучение:
- **Stage 1** — `FineTome-100k`: научить модель следовать инструкциям
- **Stage 2** — `glaive-function-calling-v2`: научить модель вызывать tools

**Инструменты:** `text_formatter`, `structure_analyzer`, `wikipedia_search` (реальный API)

**Интеграция:** нативный tool calling — модель сама генерирует `<tool_call>`, код исполняет

In [1]:
%pip install -q peft trl datasets transformers evaluate rouge_score langchain torch


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, re, json, warnings, requests
from collections import Counter
import numpy as np
import torch

from langchain.tools import tool

warnings.filterwarnings('ignore')

if torch.backends.mps.is_available():
    device = 'mps'
elif torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'

print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')

Device: mps
PyTorch: 2.10.0


## Часть 1: Fine-tuning
### 1.1 Загрузка базовой модели

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = 'Qwen/Qwen3-1.7B'  # BASE — без instruction tuning
STAGE1_DIR = './lora_stage1'
STAGE2_DIR = './lora_stage2'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.float32, trust_remote_code=True
)
base_model = base_model.to(device)
print(f'Parameters: {sum(p.numel() for p in base_model.parameters()):,}')
print(f'Loaded on: {device}')

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/622M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Parameters: 1,720,574,976
Loaded on: mps


### 1.2 Baseline тест — BASE модель до обучения

In [5]:
def generate_base_response(model, tokenizer, prompt, max_new_tokens=100):
    """Простое продолжение текста — поведение BASE модели до fine-tuning."""
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=256)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, repetition_penalty=1.3,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

def generate_response(model, tokenizer, prompt, max_new_tokens=200):
    """Инференс через chat template — для дообученной модели."""
    messages = [{'role': 'user', 'content': prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=True, temperature=0.7, top_p=0.9,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

In [6]:
TEST_PROMPTS = [
    'Write a step-by-step guide to learn Python. Use numbered steps.',
    'Explain the advantages of LoRA for fine-tuning LLMs. Use bullet points.',
]

print('=' * 60)
print('BASELINE — BASE модель БЕЗ fine-tuning')
print('Ожидаем: продолжение текста, не следует инструкции')
print('=' * 60)
for i, prompt in enumerate(TEST_PROMPTS, 1):
    print(f'\n[{i}] {prompt[:70]}')
    print(generate_base_response(base_model, tokenizer, prompt))
    print('-' * 40)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


BASELINE — BASE модель БЕЗ fine-tuning
Ожидаем: продолжение текста, не следует инструкции

[1] Write a step-by-step guide to learn Python. Use numbered steps.
 Make sure the instructions are clear and concise, with explanations for each part.

Also include at least one example of code that demonstrates how you can use your knowledge in practice.
Sure! Here's a **step-by-step guide** on learning Python:

---

### Step 1: Install Python
- Download and install [Python](https://www.python.org/downloads/) from the official website.
- Choose the appropriate version (3.x) based on what’s needed by your projects or libraries.
- Verify installation
----------------------------------------

[2] Explain the advantages of LoRA for fine-tuning LLMs. Use bullet points
 10 items.
Sure! Here are ten key advantages of using **LoRa (Low-Rank Adaptation)** for fine-tuning Large Language Models:

- **Efficiency**: LoRa allows you to train a smaller, low-rank matrix that is applied as an additional layer o

Модель достаточно продвинутая - пробует дополнить контекст и выглядит как будто пытается следовать инструкциям

In [14]:
# Та же BASE модель, те же промпты — через chat template (честное сравнение)
print('=' * 60)
print('BASELINE — BASE модель через chat template (условия = дообученным моделям)')
print('=' * 60)
for i, prompt in enumerate(TEST_PROMPTS, 1):
    print(f'\n[{i}] {prompt[:70]}')
    print(generate_response(base_model, tokenizer, prompt))
    print('-' * 40)

BASELINE — BASE модель через chat template (условия = дообученным моделям)

[1] Write a step-by-step guide to learn Python. Use numbered steps.
<think>
Okay, the user wants a step-by-step guide to learn Python. Let me start by thinking about the essential steps someone should take. First, they need to know why they want to learn Python. Maybe that's part of the first step. But the user might not need that as a separate step. Wait, the question says "step-by-step guide," so I should structure it clearly.

First, they need to install Python. So step 1 would be installing Python. Then, setting up an environment. Maybe using an IDE like PyCharm or VSCode. Next, learning basic syntax. Variables, data types, control structures. Then moving on to functions and modules. Oh, and maybe practicing with projects. Also, online resources like Codecademy or Coursera. Debugging is important too. And building projects to apply knowledge. Finally, advanced topics like OOP, web development, data analysis

## Stage 1: FineTome-100k — Instruction Following

Обучаем BASE модель следовать инструкциям на высококачественных примерах.

In [7]:
from datasets import load_dataset

print('Loading FineTome-100k...')
finetome_ds = load_dataset('mlabonne/FineTome-100k', split='train')
print(f'Size: {len(finetome_ds):,}')

# Анализ поля score (качество примеров)
scores = finetome_ds['score']
print(f'Score — min: {min(scores):.2f}, max: {max(scores):.2f}, mean: {np.mean(scores):.2f}')
print(f'Score >= 4.5: {sum(1 for s in scores if s >= 4.5):,}')

# Берём топ-2200 по score
TRAIN1, EVAL1 = 2000, 200
top_ds = finetome_ds.sort('score', reverse=True).select(range(TRAIN1 + EVAL1))
print(f'Score в выборке: min={min(top_ds["score"]):.2f}, max={max(top_ds["score"]):.2f}')

Loading FineTome-100k...


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 0ac63093-ee50-4880-8357-b349564f9a3b)')' thrown while requesting HEAD https://huggingface.co/datasets/mlabonne/FineTome-100k/resolve/main/README.md
Retrying in 1s [Retry 1/5].


Size: 100,000
Score — min: 3.74, max: 5.21, mean: 3.97
Score >= 4.5: 1,435
Score в выборке: min=4.45, max=5.21


In [8]:
def format_conversation(example):
    """ShareGPT диалог → Qwen3 chat template."""
    conversations = example.get('conversations', [])
    if not conversations:
        return {'text': ''}
    messages = []
    for turn in conversations:
        role = {'system': 'system', 'human': 'user', 'gpt': 'assistant'}.get(turn['from'])
        if role:
            messages.append({'role': role, 'content': turn['value']})
    if not messages:
        return {'text': ''}
    return {'text': tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

In [9]:
print('Форматируем...')
ft_subset = top_ds.map(format_conversation, remove_columns=top_ds.column_names)
ft_subset = ft_subset.filter(lambda x: len(x['text']) > 100)
ft_train = ft_subset.select(range(min(TRAIN1, len(ft_subset) - EVAL1)))
ft_eval  = ft_subset.select(range(len(ft_subset) - EVAL1, len(ft_subset)))
print(f'Train: {len(ft_train)}, Eval: {len(ft_eval)}')
print(ft_train[0]['text'][:300])

Форматируем...


Map:   0%|          | 0/2200 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2200 [00:00<?, ? examples/s]

Train: 2000, Eval: 200
<|im_start|>user
Explain what boolean operators are, what they do, and provide examples of how they can be used in programming. Additionally, describe the concept of operator precedence and provide examples of how it affects the evaluation of boolean expressions. Discuss the difference between short


### LoRA конфиг и обучение Stage 1

In [15]:
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

os.environ['PYTORCH_MPS_HIGH_WATERMARK_RATIO'] = '0.0'
os.makedirs(STAGE1_DIR, exist_ok=True)

lora_cfg = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj'],
    lora_dropout=0.05, bias='none', task_type='CAUSAL_LM'
)
peft_model = get_peft_model(base_model, lora_cfg)
trainable, total = peft_model.get_nb_trainable_parameters()
print(f'Trainable: {trainable:,} ({100*trainable/total:.2f}%)')

trainer1 = SFTTrainer(
    model=peft_model,
    args=SFTConfig(
        output_dir=STAGE1_DIR, num_train_epochs=1,
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        learning_rate=2e-4, warmup_ratio=0.05, lr_scheduler_type='cosine',
        logging_steps=25, eval_steps=250, eval_strategy='steps',
        load_best_model_at_end=True, metric_for_best_model='eval_loss',
        dataset_text_field='text', max_length=512,
        packing=False, report_to='none',
    ),
    train_dataset=ft_train,
    eval_dataset=ft_eval,
)

Trainable: 6,881,280 (0.40%)


Adding EOS to train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

In [16]:
print(f'Device: {next(trainer1.model.parameters()).device}')
print(f'Шагов: {len(ft_train) // 8}')
print('Starting Stage 1 training...')
res1 = trainer1.train()
print(f'Done! loss={res1.metrics["train_loss"]:.4f}, time={res1.metrics["train_runtime"]:.0f}s')

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Device: mps:0
Шагов: 250
Starting Stage 1 training...


Step,Training Loss,Validation Loss
250,0.850500,0.887390


'(ProtocolError('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer')), '(Request ID: b02e7db0-7655-4be7-8aaa-acb80ed16af9)')' thrown while requesting HEAD https://huggingface.co/Qwen/Qwen3-1.7B/resolve/main/config.json
Retrying in 1s [Retry 1/5].


Done! loss=0.9697, time=6019s


In [17]:
trainer1.model.save_pretrained(STAGE1_DIR)
tokenizer.save_pretrained(STAGE1_DIR)
print(f'Stage 1 saved: {STAGE1_DIR}')

Stage 1 saved: ./lora_stage1


In [18]:
from peft import PeftModel

# Тест после Stage 1
stage1_model = PeftModel.from_pretrained(base_model, STAGE1_DIR, is_trainable=False).to(device)
print('\n' + '=' * 60)
print('ПОСЛЕ Stage 1 (instruction following)')
print('=' * 60)
for i, prompt in enumerate(TEST_PROMPTS, 1):
    print(f'\n[{i}] {prompt[:70]}')
    print(generate_response(stage1_model, tokenizer, prompt))
    print('-' * 40)


ПОСЛЕ Stage 1 (instruction following)

[1] Write a step-by-step guide to learn Python. Use numbered steps.
<think>

</think>

1. Start by installing Python from the official website (https://www.python.org/downloads/).
2. Open a terminal or command prompt and type "python --version" to confirm that Python is installed.
3. Create a new file using a text editor, such as Notepad or Sublime Text, and save it with a .py extension (e.g., hello.py).
4. Write code in the file, for example: 
   ```python
   print("Hello, world!")
   ```
5. Save the file and double-click it to run the script.
6. You should see the output of your code displayed on the screen.

7. To write more complex programs, you can use basic data types like integers, floats, strings, and booleans.
8. You can also create variables to store values:
   ```python
   name = "Alice"
   age = 30
   ```

9. Combine multiple lines of code using triple quotes
----------------------------------------

[2] Explain the advantages of LoRA

Модель обучилась следовать инструкциям. Но не обучалась заполнять think

### Merge LoRA Stage 1 → подготовка к Stage 2

`merge_and_unload()` вшивает веса LoRA в base модель. Это необходимо перед Stage 2 — нельзя накладывать два независимых LoRA одновременно.

In [19]:
print('Merging Stage 1 LoRA into base model...')
merged_model = stage1_model.merge_and_unload()
merged_model = merged_model.to(device)
print(f'Merged. Parameters: {sum(p.numel() for p in merged_model.parameters()):,}')
print('Готов к Stage 2')

Merging Stage 1 LoRA into base model...
Merged. Parameters: 1,720,574,976
Готов к Stage 2


## Stage 2: Glaive Function Calling

Дообучаем merged_model на примерах с вызовом функций. Модель научится генерировать `<tool_call>` когда нужна внешняя функция.

In [20]:
from datasets import load_dataset

print('Loading glaive-function-calling-v2...')
glaive_ds = load_dataset('glaiveai/glaive-function-calling-v2', split='train')
print(f'Size: {len(glaive_ds):,}')
print(f'Features: {list(glaive_ds.features.keys())}')

# Пример с реальным function call
for i, ex in enumerate(glaive_ds):
    if '<functioncall>' in ex['chat']:
        print(f'\nПример #{i} с functioncall:')
        print(repr(ex['chat'][:400]))
        break

Loading glaive-function-calling-v2...
Size: 112,960
Features: ['system', 'chat']

Пример #1 с functioncall:
'USER: Can you tell me the latest news headlines for the United States?\n\n\nASSISTANT: <functioncall> {"name": "get_news_headlines", "arguments": \'{"country": "United States"}\'} <|endoftext|>\n\n\nFUNCTION RESPONSE: {"headlines": ["Biden announces new vaccine mandates", "Hurricane Ida devastates Louisiana", "Apple unveils new iPhone", "NASA\'s Perseverance rover collects first Mars rock sample"]}\n\n\nASSIS'


In [21]:
def parse_glaive_to_messages(example):
    """
    Конвертирует glaive формат в Qwen3 messages.
    ASSISTANT: <functioncall>{} → assistant: <tool_call>\n{}\n</tool_call>
    FUNCTION RESPONSE: {} → user: [TOOL RESPONSE]\n{}
    """
    system_raw = example['system'].replace('SYSTEM:', '').strip()
    parts = re.split(r'\n{2,}', example['chat'].strip())
    messages = [{'role': 'system', 'content': system_raw}]
    for part in parts:
        part = part.strip()
        if part.startswith('USER:'):
            messages.append({'role': 'user', 'content': part[5:].strip()})
        elif part.startswith('FUNCTION RESPONSE:'):
            content = part[18:].strip().replace('<|endoftext|>', '').strip()
            messages.append({'role': 'user', 'content': f'[TOOL RESPONSE]\n{content}'})
        elif part.startswith('ASSISTANT:'):
            content = part[10:].strip().replace('<|endoftext|>', '').strip()
            if '<functioncall>' in content:
                m = re.search(r'<functioncall>\s*(\{.*\})', content, re.DOTALL)
                if m:
                    messages.append({'role': 'assistant', 'content': f'<tool_call>\n{m.group(1)}\n</tool_call>'})
            elif content:
                messages.append({'role': 'assistant', 'content': content})
    return messages

In [22]:
# Проверка
msgs = parse_glaive_to_messages(glaive_ds[1])
print(f'Сообщений: {len(msgs)}')
for m in msgs:
    print(f'[{m["role"]}]: {m["content"][:120]}')

Сообщений: 9
[system]: You are a helpful assistant with access to the following functions. Use them if required -
{
    "name": "get_news_headl
[user]: Can you tell me the latest news headlines for the United States?
[assistant]: <tool_call>
{"name": "get_news_headlines", "arguments": '{"country": "United States"}'}
</tool_call>
[user]: [TOOL RESPONSE]
{"headlines": ["Biden announces new vaccine mandates", "Hurricane Ida devastates Louisiana", "Apple unve
[assistant]: Here are the latest news headlines for the United States:
1. Biden announces new vaccine mandates
2. Hurricane Ida devas
[user]: That's interesting. What about the news in France?
[assistant]: <tool_call>
{"name": "get_news_headlines", "arguments": '{"country": "France"}'}
</tool_call>
[user]: [TOOL RESPONSE]
{"headlines": ["France recalls ambassadors to US and Australia", "French election: Macron's party braces
[assistant]: Here are the latest news headlines for France:
1. France recalls ambassadors to US and Australia
2

In [23]:
def format_glaive_example(example):
    messages = parse_glaive_to_messages(example)
    try:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        return {'text': text}
    except Exception:
        return {'text': ''}

print('Фильтруем примеры с function calls...')
fc_ds = glaive_ds.filter(lambda x: '<functioncall>' in x['chat'])
print(f'С function calls: {len(fc_ds):,} из {len(glaive_ds):,}')

TRAIN2, EVAL2 = 2000, 200
fc_subset = fc_ds.select(range(TRAIN2 + EVAL2))
fc_subset = fc_subset.map(format_glaive_example, remove_columns=fc_subset.column_names)
fc_subset = fc_subset.filter(lambda x: len(x['text']) > 100)

# Фильтр по длине токенов — без обрезания tool call последовательностей
fc_subset = fc_subset.filter(lambda x: len(tokenizer.encode(x['text'])) <= 512)

fc_train = fc_subset.select(range(min(TRAIN2, len(fc_subset) - EVAL2)))
fc_eval  = fc_subset.select(range(len(fc_subset) - EVAL2, len(fc_subset)))
print(f'Train: {len(fc_train)}, Eval: {len(fc_eval)}')

Фильтруем примеры с function calls...
С function calls: 63,218 из 112,960
Train: 1745, Eval: 200


### LoRA конфиг и обучение Stage 2

In [24]:
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

os.makedirs(STAGE2_DIR, exist_ok=True)

peft_model2 = get_peft_model(merged_model, LoraConfig(
    r=8, lora_alpha=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj'],
    lora_dropout=0.05, bias='none', task_type='CAUSAL_LM'
))
trainable, total = peft_model2.get_nb_trainable_parameters()
print(f'Trainable: {trainable:,} ({100*trainable/total:.2f}%)')

trainer2 = SFTTrainer(
    model=peft_model2,
    args=SFTConfig(
        output_dir=STAGE2_DIR, num_train_epochs=1,
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        learning_rate=1e-4, warmup_ratio=0.05, lr_scheduler_type='cosine',
        logging_steps=25, eval_steps=50, eval_strategy='steps',
        load_best_model_at_end=True, metric_for_best_model='eval_loss',
        dataset_text_field='text', max_length=512,
        packing=False, report_to='none',
    ),
    train_dataset=fc_train,
    eval_dataset=fc_eval,
)

Trainable: 6,881,280 (0.40%)


Adding EOS to train dataset:   0%|          | 0/1745 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1745 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

In [25]:
print(f'Device: {next(trainer2.model.parameters()).device}')
print(f'Шагов: {len(fc_train) // 8}')
print('Starting Stage 2 training...')
res2 = trainer2.train()
print(f'Done! loss={res2.metrics["train_loss"]:.4f}, time={res2.metrics["train_runtime"]:.0f}s')

Device: mps:0
Шагов: 218
Starting Stage 2 training...


Step,Training Loss,Validation Loss
50,0.329000,0.279956
100,0.237000,0.219265
150,0.203500,0.198294
200,0.197300,0.193318


Done! loss=0.2790, time=43659s


In [26]:
trainer2.model.save_pretrained(STAGE2_DIR)
tokenizer.save_pretrained(STAGE2_DIR)
print(f'Stage 2 saved: {STAGE2_DIR}')

Stage 2 saved: ./lora_stage2


In [27]:
from peft import PeftModel

# Загружаем финальную модель
final_model = PeftModel.from_pretrained(merged_model, STAGE2_DIR, is_trainable=False).to(device)
print('Final model loaded')

# Тест: попросим вызвать функцию
test_fc_prompt = (
    'You have access to this function: '
    '{"name": "get_weather", "description": "Get weather", '
    '"parameters": {"city": {"type": "string"}}}. '
    'What is the weather in Paris?'
)
print('\n' + '=' * 60)
print('ПОСЛЕ Stage 2 (function calling)')
print('=' * 60)
print(f'Prompt: {test_fc_prompt[:80]}...')
print(generate_response(final_model, tokenizer, test_fc_prompt))

Final model loaded

ПОСЛЕ Stage 2 (function calling)
Prompt: You have access to this function: {"name": "get_weather", "description": "Get we...
<tool_call>
{"name": "get_weather", "arguments": '{"city": "Paris"}'}
</tool_call>


In [28]:
import evaluate

rouge = evaluate.load('rouge')

EVAL_PAIRS = [
    ('List 3 tips for writing clean code.', 'Use meaningful names. Keep functions small. Write tests.'),
    ('What are the main features of Python?', 'Simple syntax. Dynamic typing. Large library ecosystem.'),
    ('Explain what is an API.', 'API allows software to communicate. It uses standard protocols.'),
    ('Give steps to debug a Python program.', 'Read the error. Add print statements. Use a debugger.'),
    ('What are benefits of version control?', 'Track changes. Collaborate with team. Revert versions.'),
]

def eval_rouge(model, pairs):
    preds, refs = [], []
    for prompt, ref in pairs:
        preds.append(generate_response(model, tokenizer, prompt, max_new_tokens=80))
        refs.append(ref)
    return rouge.compute(predictions=preds, references=refs, use_stemmer=True)

print('Вычисляем ROUGE...')
s_base   = eval_rouge(base_model,   EVAL_PAIRS)
s_stage1 = eval_rouge(stage1_model, EVAL_PAIRS)
s_final  = eval_rouge(final_model,  EVAL_PAIRS)

print(f'\n{"Метрика":<12} {"Base":>8} {"Stage1":>8} {"Final":>8}')
print('-' * 42)
for k in sorted(s_base.keys()):
    print(f'{k:<12} {s_base[k]:>8.4f} {s_stage1[k]:>8.4f} {s_final[k]:>8.4f}')

Вычисляем ROUGE...

Метрика          Base   Stage1    Final
------------------------------------------
rouge1         0.1283   0.1215   0.1163
rouge2         0.0249   0.0210   0.0271
rougeL         0.1217   0.1085   0.0951
rougeLsum      0.1231   0.1011   0.0955


по метрике ROUGE не видно изменений, но зато они видны по результатам вызовов модели

## Часть 2: LangChain Tools

1. **`text_formatter`** — форматирует текст (bullets, numbered, summary, sections)
2. **`structure_analyzer`** — анализирует качество текста, quality score 0-100
3. **`wikipedia_search`** — реальный Wikipedia REST API (без API-ключа)

In [29]:
@tool
def text_formatter(input_json: str) -> str:
    """
    Formats text into a structured layout.
    Input: JSON string with 'text' and 'format_type'.
    format_type: 'bullet_points', 'numbered_list', 'summary', 'sections'.
    Example: '{"text": "Python is great.", "format_type": "bullet_points"}'
    """
    try:
        data = json.loads(input_json)
        text, fmt = data.get('text', ''), data.get('format_type', 'bullet_points')
    except (json.JSONDecodeError, TypeError):
        text, fmt = str(input_json), 'bullet_points'
    sentences = [s.strip() for s in re.split(r'[.!?]+', text) if len(s.strip()) > 10]
    if not sentences:
        return text
    if fmt == 'bullet_points':
        return '\n'.join(f'• {s}' for s in sentences)
    elif fmt == 'numbered_list':
        return '\n'.join(f'{i+1}. {s}' for i, s in enumerate(sentences))
    elif fmt == 'summary':
        return '. '.join(sentences[:3]) + '.'
    elif fmt == 'sections':
        chunks = [sentences[i:i+3] for i in range(0, len(sentences), 3)]
        result = []
        for j, chunk in enumerate(chunks[:4], 1):
            result.append(f'\n### Section {j}')
            result.extend([f'  - {s}' for s in chunk])
        return '\n'.join(result)
    return text

In [30]:
# Тест
sample = 'Python is easy. It has great libraries. Many companies use it.'
for fmt in ['bullet_points', 'numbered_list', 'summary']:
    print(f'[{fmt}]')
    print(text_formatter.run(json.dumps({'text': sample, 'format_type': fmt})))
    print()

[bullet_points]
• Python is easy
• It has great libraries
• Many companies use it

[numbered_list]
1. Python is easy
2. It has great libraries
3. Many companies use it

[summary]
Python is easy. It has great libraries. Many companies use it.



In [31]:
@tool
def structure_analyzer(text: str) -> str:
    """
    Analyzes structure and quality of text.
    Returns JSON with word count, vocabulary richness, readability,
    structure indicators, and quality score (0-100).
    """
    words = text.split()
    if not words:
        return json.dumps({'error': 'Empty text'})
    sentences = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
    unique_words = len(set(w.lower().strip('.,!?;:') for w in words))
    vocab_richness = round(unique_words / len(words), 3)
    avg_sent_len = round(len(words) / len(sentences), 1) if sentences else 0
    syllables = sum(max(1, len(re.findall(r'[aeiouAEIOU]', w))) for w in words)
    flesch = 206.835 - 1.015 * avg_sent_len - 84.6 * (syllables / len(words))
    has_bullets  = bool(re.search(r'^[•\-\*]\s', text, re.MULTILINE))
    has_numbered = bool(re.search(r'^\d+[\.)]', text, re.MULTILINE))
    has_headers  = bool(re.search(r'^#{1,3}\s', text, re.MULTILINE))
    structure_bonus = (15 if has_bullets or has_numbered else 0) + (10 if has_headers else 0)
    quality_score = min(100, int(vocab_richness * 50) + structure_bonus + min(25, len(sentences) * 3))
    return json.dumps({
        'words': len(words), 'unique_words': unique_words,
        'vocab_richness': vocab_richness,
        'readability': 'Easy' if flesch > 70 else ('Medium' if flesch > 50 else 'Complex'),
        'has_bullets': has_bullets, 'has_numbered': has_numbered, 'has_headers': has_headers,
        'quality_score': quality_score
    }, indent=2)

In [32]:
# Тест
r = json.loads(structure_analyzer.run('Great.\n\n• Point 1\n• Point 2\n• Point 3'))
print(f'Quality: {r["quality_score"]}/100, bullets={r["has_bullets"]}')

Quality: 51/100, bullets=True


In [36]:
@tool
def wikipedia_search(query: str) -> str:
    """
    Searches Wikipedia using real REST API (no API key required).
    Returns a 4-sentence summary of the most relevant article.
    """
    base_url = 'https://en.wikipedia.org/w/api.php'
    headers = {'User-Agent': 'lora-homework/1.0 (educational project)'}
    try:
        hits = requests.get(base_url, headers=headers, params={
            'action': 'query', 'list': 'search', 'srsearch': query,
            'srlimit': 1, 'format': 'json'
        }, timeout=10).json().get('query', {}).get('search', [])
        if not hits:
            return f'No article found for: {query}'
        pages = requests.get(base_url, headers=headers, params={
            'action': 'query', 'pageids': hits[0]['pageid'],
            'prop': 'extracts', 'exintro': True, 'explaintext': True,
            'exsentences': 4, 'format': 'json'
        }, timeout=10).json().get('query', {}).get('pages', {})
        page = list(pages.values())[0]
        extract = re.sub(r'\n+', ' ', page.get('extract', '')).strip()
        return f'[Wikipedia] {page.get("title", "")}\n\n{extract[:600]}'
    except requests.exceptions.Timeout:
        return 'Wikipedia search timed out.'
    except Exception as e:
        return f'Error: {e}'

In [37]:
# Тест — реальный API
print(wikipedia_search.run('LoRA low-rank adaptation fine-tuning')[:300])

[Wikipedia] Fine-tuning (deep learning)

Fine-tuning (in deep learning) is the process of adapting a model trained for one task (the upstream task) to perform a different, usually more specific, task (the downstream task). It is considered a form of transfer learning, as it reuses knowledge learned 


## Часть 3: Интеграция — нативный Function Calling

Модель сама генерирует `<tool_call>` → парсит код → вызывает Python-функцию → возвращает результат модели → модель формирует финальный ответ.

In [ ]:
TOOLS_REGISTRY = {
    'text_formatter': text_formatter,
    'structure_analyzer': structure_analyzer,
    'wikipedia_search': wikipedia_search,
}

TOOLS_DESCRIPTION = json.dumps([
    {
        'name': 'text_formatter',
        'description': 'Formats SHORT existing text (max 2-3 sentences) into structured layout. Do NOT generate long content inside arguments — write brief text first, then pass it here.',
        'parameters': {
            'text': 'The text to format (string)',
            'format_type': 'One of: bullet_points, numbered_list, summary, sections'
        },
        'example': {"name": "text_formatter", "arguments": {"text": "Python is fast. It is easy.", "format_type": "bullet_points"}}
    },
    {
        'name': 'structure_analyzer',
        'description': 'Analyzes text quality and structure. Returns quality score 0-100.',
        'parameters': {'text': 'The text to analyze (string)'},
        'example': {"name": "structure_analyzer", "arguments": {"text": "Some text to analyze."}}
    },
    {
        'name': 'wikipedia_search',
        'description': 'Searches Wikipedia for factual information.',
        'parameters': {'query': 'Search query (string)'},
        'example': {"name": "wikipedia_search", "arguments": {"query": "transformer neural network"}}
    },
], indent=2)

def parse_tool_call(raw):
    """Парсит tool call JSON, включая arguments как строку с одинарными кавычками."""
    import ast
    # Убираем '} в конце → } (артефакт генерации: "value"'})
    raw = raw.replace(chr(34) + chr(39) + chr(125), chr(34) + chr(125) + chr(39))  # "'}  ->  "}'
    # Попытка 1: стандартный JSON
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        pass
    # Попытка 2: ast.literal_eval понимает одинарные кавычки (Python dict)
    try:
        return ast.literal_eval(raw)
    except Exception:
        pass
    # Попытка 3: извлекаем name и arguments по отдельности через regex
    name_m = re.search(r'"name"\s*:\s*"([^"]+)"', raw)
    args_m = re.search(r'"arguments"\s*:\s*\'(.*?)\'(?=\s*\}?\s*$)', raw, re.DOTALL)
    if name_m:
        name = name_m.group(1)
        if args_m:
            try:
                args = json.loads(args_m.group(1))
            except Exception:
                args = {'value': args_m.group(1)}
        else:
            args = {}
        return {'name': name, 'arguments': args}
    raise ValueError(f'Cannot parse tool call: {raw[:100]}')

def run_with_tools(model, tokenizer, user_input, verbose=True):
    """
    Inference loop с нативным tool calling:
    1. Модель получает запрос + описания tools
    2. Если генерирует <tool_call> — вызываем функцию, возвращаем результат
    3. Модель формирует финальный ответ
    """
    system_prompt = (
        f'You are a helpful assistant. You have access to these tools:\n{TOOLS_DESCRIPTION}\n'
        'IMPORTANT rules:\n'
        '1. To use a tool wrap it in <tool_call> tags:\n'
        '<tool_call>\n{"name": "tool_name", "arguments": {"key": "value"}}\n</tool_call>\n'
        '2. Always use double quotes in JSON. Never use single quotes.\n'
        '3. Keep arguments short and concise — do not put long text inside tool arguments.\n'
        '4. Always wrap tool calls in <tool_call> tags, even if calling a second tool.'
    )
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user',   'content': user_input},
    ]

    if verbose:
        print(f'User: {user_input}')
        print('-' * 50)

    last_tool_result = ''
    for step in range(4):
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=1024).to(device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=1000, do_sample=True,
                                 temperature=0.7, top_p=0.9, repetition_penalty=1.1,
                                 eos_token_id=tokenizer.eos_token_id,
                                 pad_token_id=tokenizer.pad_token_id)
        response = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

        # Нормализуем tool_call теги
        if '<tool_call>' in response and '</tool_call>' not in response:
            response += '\n</tool_call>'  # незакрытый тег
        if '</tool_call>' in response and '<tool_call>' not in response:
            response = '<tool_call>\n' + response  # отсутствует открывающий тег
        tc_match = re.search(r'<tool_call>\s*(.*?)\s*</tool_call>', response, re.DOTALL)
        if tc_match:
            try:
                call = parse_tool_call(tc_match.group(1).strip())
                tool_name = call.get('name', '')
                tool_args = call.get('arguments', {})
                # arguments может быть строкой JSON — распарсим
                if isinstance(tool_args, str):
                    try:
                        tool_args = json.loads(tool_args)
                    except Exception:
                        import ast
                        try:
                            tool_args = ast.literal_eval(tool_args)
                        except Exception:
                            tool_args = {'value': tool_args}
                if verbose:
                    print(f'[tool_call] {tool_name}({tool_args})')
                if tool_name in TOOLS_REGISTRY:
                    # Если аргументы пустые — используем последний tool_result
                    if not tool_args and last_tool_result:
                        tool_args = {'text': last_tool_result}
                    if tool_args:
                        # Unwrap {'value': '{"text": "...", ...}'} → {'text': '...', ...}
                        if list(tool_args.keys()) == ['value'] and isinstance(tool_args['value'], str):
                            import ast
                            raw_val = tool_args['value']
                            parsed = None
                            for attempt in [
                                raw_val,
                                raw_val.replace('\n', '\\n').replace('\t', '\\t'),
                            ]:
                                try:
                                    parsed = json.loads(attempt)
                                    break
                                except Exception:
                                    try:
                                        parsed = ast.literal_eval(attempt)
                                        break
                                    except Exception:
                                        pass
                            if parsed and isinstance(parsed, dict):
                                tool_args = parsed
                        if tool_name == 'text_formatter':
                            # text_formatter принимает весь JSON {"text": "...", "format_type": "..."}
                            arg_val = json.dumps(tool_args)
                        else:
                            # Остальные инструменты принимают первый аргумент как строку
                            arg_val = list(tool_args.values())[0]
                    else:
                        arg_val = ''
                    tool_result = TOOLS_REGISTRY[tool_name].run(arg_val)
                    last_tool_result = str(tool_result)
                    if verbose:
                        print(f'[tool_result] {str(tool_result)[:150]}')
                    messages.append({'role': 'assistant', 'content': response})
                    messages.append({'role': 'user', 'content': f'[TOOL RESPONSE]\n{tool_result}'})
                    continue
            except Exception as e:
                if verbose:
                    print(f'[tool_call parse error] {e}')

        if verbose:
            print(f'\nFinal Response:\n{"=" * 50}')
        return response

    return response


In [115]:
scenarios = [
    'What is the transformer architecture in deep learning?',
    'Write a guide to Python virtual environments. Use numbered steps.',
    'List the best practices for clean code and analyze the quality of your response.',
]

for i, scenario in enumerate(scenarios, 1):
    print(f'\n{"=" * 60}')
    print(f'СЦЕНАРИЙ {i}')
    print('=' * 60)
    result = run_with_tools(final_model, tokenizer, scenario, verbose=True)
    print(result)


СЦЕНАРИЙ 1
User: What is the transformer architecture in deep learning?
--------------------------------------------------
[tool_call] wikipedia_search({'query': 'transformer architecture'})
[tool_result] [Wikipedia] Transformer (deep learning)

In deep learning, the transformer is a family of artificial neural network architectures based on the multi-h

Final Response:
<think>

</think>

The transformer architecture is a type of deep learning model that uses an attention mechanism to process input data. This allows the model to focus on relevant parts of the input when generating output.

СЦЕНАРИЙ 2
User: Write a guide to Python virtual environments. Use numbered steps.
--------------------------------------------------
[tool_call] text_formatter({'text': 'Creating a Python virtual environment involves several steps.', 'format_type': 'numbered_list'})
[tool_result] 1. Creating a Python virtual environment involves several steps

Final Response:
<think>

</think>

1. First, open your 

Самый стабильный первый сценарий. Для второго и третьего модель редко делает корректный вызов тулов - то порядок кавычек путается, то неправильные переводы строки (задваивается слэш), то и вовсе не удается сгенерировать аргументы. То есть модель подходит для вызова тулов либо без аргументов, либо с короткими и простыми аргументами.

## Многошаговый tool calling

Модель вызывает инструменты последовательно.

In [118]:
print('Multi-step tool calling')
print('=' * 60)
result = run_with_tools(
    final_model, tokenizer,
    'Find information about BERT model, then format the key points as bullet points.',
    verbose=True
)
print(result)

Multi-step tool calling
User: Find information about BERT model, then format the key points as bullet points.
--------------------------------------------------
[tool_call] wikipedia_search({'query': 'BERT model'})
[tool_result] [Wikipedia] BERT (language model)

Bidirectional encoder representations from transformers (BERT) is a language model introduced in October 2018 by re
[tool_call] text_formatter({})
[tool_result] • [Wikipedia] BERT (language model)

Bidirectional encoder representations from transformers (BERT) is a language model introduced in October 2018 by 

Final Response:
<think>

</think>

I've found some information about the BERT model. Here's a formatted version of the key points:

- Bidirectional encoder representations from transformers (BERT) is a language model introduced in October 2018 by researchers at Google.
- It learns to represent text as a sequence of vectors using self-supervised learning.
- It uses the encoder-only transformer architecture.
- BERT dramat

## Выводы

### Что реализовано

**Fine-tuning (двухэтапный):**
- **Stage 1** — `Qwen3-1.7B` BASE → FineTome-100k (топ-2000 по score) → instruction following
- **Stage 2** — merged Stage1 → glaive-function-calling-v2 (~1750 примеров) → tool calling
- Между этапами: `merge_and_unload()` — LoRA Stage 1 вшит в веса

**LangChain Tools:**
- `text_formatter` — 4 режима форматирования
- `structure_analyzer` — quality score 0-100
- `wikipedia_search` — реальный Wikipedia REST API

**Интеграция:**
- Нативный tool calling: модель генерирует `<tool_call>`, код исполняет, результат возвращается модели
- Inference loop с 3 шагами tool calling

### Ограничения
- 2000 примеров на этап — минимум; production требует 50k+
- Для production-grade tool calling нужна модель 7B+ и специализированный датасет